# 01b · Time-scale sensitivity (days / months / semesters / years)

**Claim:** re-scaling `time` is a linear reparameterisation — inference (p-values, CIs on contrasts) is invariant. Coefficient magnitudes change by the scale factor. Coarser scales tend to improve numerical convergence.

This notebook refits the same LMM at four time scales and prints the four contrast tables side-by-side.

In [1]:
source("helpers/pain_helpers.R")
suppressPackageStartupMessages({
  library(lme4); library(emmeans); library(purrr); library(dplyr)
})
df_long <- readRDS(file.path(OUT_OBJ, "pain_long.rds"))

# Same exclusion rule as notebook 01
dbs_ids <- df_long %>% dplyr::filter(traj == "Post-DBS") %>% dplyr::pull(PATNO) %>% unique()
exclude_ids <- df_long %>%
  dplyr::filter(traj == "Pre-DBS", PATNO %in% dbs_ids) %>%
  dplyr::group_by(PATNO) %>%
  dplyr::summarise(max_pre_months = max(time_pos_months, na.rm = TRUE), .groups = "drop") %>%
  dplyr::filter(max_pre_months > 60) %>% dplyr::pull(PATNO)

base <- df_long %>%
  dplyr::filter(!PATNO %in% exclude_ids)

# Add time at each scale
base <- base %>% dplyr::mutate(
  time_days      = time_pos,
  time_months    = time_pos_months,
  time_semesters = time_pos_months / 6,
  time_years     = time_pos_months / 12
)
cat("Rows:", nrow(base), "  Patients:", dplyr::n_distinct(base$PATNO), "\n")

Rows: 6206   Patients: 167 


In [2]:
fit_one <- function(time_var) {
  f <- as.formula(sprintf("NP1PAIN ~ %s * traj + (1 + %s | PATNO)", time_var, time_var))
  m <- lme4::lmer(
    f, data = base, weights = weight_sw_trim90, REML = FALSE,
    control = lme4::lmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5))
  )
  grad <- max(abs(m@optinfo$derivs$gradient))
  sl <- emmeans::emtrends(m, specs = "traj", var = time_var)
  ct <- as.data.frame(pairs(sl, adjust = "tukey"))
  list(model = m, grad = grad, slopes = as.data.frame(sl), contrasts = ct)
}

scales <- c("time_days","time_months","time_semesters","time_years")
fits <- stats::setNames(lapply(scales, fit_one), scales)

grad_tbl <- data.frame(
  scale = scales,
  max_abs_gradient = sapply(fits, function(x) signif(x$grad, 4))
)
print(grad_tbl)

Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model failed to converge with max|grad| = 2.23981 (tol = 0.002, component 1)”


Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
“Model is nearly unidentifiable: very large eigenvalue
 - Rescale variables?”


Note: D.f. calculations have been disabled because the number of observations exceeds 3000.
To enable adjustments, add the argument 'pbkrtest.limit = 6206' (or larger)
[or, globally, 'set emm_options(pbkrtest.limit = 6206)' or larger];
but be warned that this may result in large computation time and memory use.



Note: D.f. calculations have been disabled because the number of observations exceeds 3000.
To enable adjustments, add the argument 'lmerTest.limit = 6206' (or larger)
[or, globally, 'set emm_options(lmerTest.limit = 6206)' or larger];
but be warned that this may result in large computation time and memory use.



Note: D.f. calculations have been disabled because the number of observations exceeds 3000.
To enable adjustments, add the argument 'pbkrtest.limit = 6206' (or larger)
[or, globally, 'set emm_options(pbkrtest.limit = 6206)' or larger];
but be warned that this may result in large computation time and memory use.



...

Note: D.f. calculations have been disabled because the number of observations exceeds 3000.
To enable adjustments, add the argument 'pbkrtest.limit = 6206' (or larger)
[or, globally, 'set emm_options(pbkrtest.limit = 6206)' or larger];
but be warned that this may result in large computation time and memory use.



Note: D.f. calculations have been disabled because the number of observations exceeds 3000.
To enable adjustments, add the argument 'lmerTest.limit = 6206' (or larger)
[or, globally, 'set emm_options(lmerTest.limit = 6206)' or larger];
but be warned that this may result in large computation time and memory use.



Note: D.f. calculations have been disabled because the number of observations exceeds 3000.
To enable adjustments, add the argument 'pbkrtest.limit = 6206' (or larger)
[or, globally, 'set emm_options(pbkrtest.limit = 6206)' or larger];
but be warned that this may result in large computation time and memory use.



Note: D.f. calculations have been disabled because the number of observations exceeds 3000.
To enable adjustments, add the argument 'lmerTest.limit = 6206' (or larger)
[or, globally, 'set emm_options(lmerTest.limit = 6206)' or larger];
but be warned that this may result in large computation time and memory use.



                        scale max_abs_gradient
time_days           time_days        7.031e+02
time_months       time_months        2.555e-02
time_semesters time_semesters        1.799e-04
time_years         time_years        3.953e-05


In [3]:
# Slope estimates at each scale (note the linear rescaling)
for (s in scales) {
  cat("\n==========", s, "==========\n")
  print(fits[[s]]$slopes)
}


========== time_days ==========
 traj      time_days.trend           SE  df     asymp.LCL    asymp.UCL
 Pre-DBS     -1.723180e-04 0.0001696006 Inf -0.0005047292 0.0001600931
 Post-DBS     1.775095e-04 0.0001478401 Inf -0.0001122517 0.0004672708
 Never-DBS    8.603281e-05 0.0001326437 Inf -0.0001739441 0.0003460097

Degrees-of-freedom method: asymptotic 
Confidence level used: 0.95 

========== time_months ==========
 traj      time_months.trend          SE  df    asymp.LCL   asymp.UCL
 Pre-DBS        -0.005255277 0.005195346 Inf -0.015437968 0.004927414
 Post-DBS        0.005411785 0.004538350 Inf -0.003483218 0.014306788
 Never-DBS       0.002676972 0.004066681 Inf -0.005293577 0.010647520

Degrees-of-freedom method: asymptotic 
Confidence level used: 0.95 

========== time_semesters ==========
 traj      time_semesters.trend         SE  df   asymp.LCL  asymp.UCL
 Pre-DBS            -0.03153166 0.03117208 Inf -0.09262781 0.02956449
 Post-DBS            0.03247071 0.02723010 Inf -0.02

In [4]:
# Pairwise contrasts: p-values should be identical across scales
for (s in scales) {
  cat("\n==========", s, "==========\n")
  print(fits[[s]]$contrasts)
}


========== time_days ==========
 contrast                      estimate           SE  df z.ratio p.value
 (Pre-DBS) - (Post-DBS)   -0.0003498276 0.0001073922 Inf  -3.257  0.0032
 (Pre-DBS) - (Never-DBS)  -0.0002583509 0.0002153108 Inf  -1.200  0.4532
 (Post-DBS) - (Never-DBS)  0.0000914767 0.0001986229 Inf   0.461  0.8897

Degrees-of-freedom method: asymptotic 
P value adjustment: tukey method for comparing a family of 3 estimates 

========== time_months ==========
 contrast                     estimate          SE  df z.ratio p.value
 (Pre-DBS) - (Post-DBS)   -0.010667062 0.003268071 Inf  -3.264  0.0032
 (Pre-DBS) - (Never-DBS)  -0.007932248 0.006597690 Inf  -1.202  0.4518
 (Post-DBS) - (Never-DBS)  0.002734813 0.006093810 Inf   0.449  0.8949

Degrees-of-freedom method: asymptotic 
P value adjustment: tukey method for comparing a family of 3 estimates 

========== time_semesters ==========
 contrast                    estimate         SE  df z.ratio p.value
 (Pre-DBS) - (Post-DBS)  

In [5]:
# Clean year-scale summary for reporting
sl_yr <- fits$time_years$slopes
ct_yr <- fits$time_years$contrasts

cat("NP1PAIN change per YEAR by phase:\n")
print(sl_yr)

cat("\nPairwise Δslope (per year):\n")
print(ct_yr)

save_table(sl_yr, "lmm_slopes_per_year")
save_table(ct_yr, "lmm_slope_contrasts_per_year")

NP1PAIN change per YEAR by phase:


 traj      time_years.trend         SE  df   asymp.LCL  asymp.UCL
 Pre-DBS        -0.06306332 0.06234415 Inf -0.18525561 0.05912896
 Post-DBS        0.06494142 0.05446020 Inf -0.04179861 0.17168145
 Never-DBS       0.03212366 0.04880017 Inf -0.06352293 0.12777024

Degrees-of-freedom method: asymptotic 
Confidence level used: 0.95 



Pairwise Δslope (per year):


 contrast                    estimate         SE  df z.ratio p.value
 (Pre-DBS) - (Post-DBS)   -0.12800474 0.03921685 Inf  -3.264  0.0032
 (Pre-DBS) - (Never-DBS)  -0.09518698 0.07917228 Inf  -1.202  0.4518
 (Post-DBS) - (Never-DBS)  0.03281776 0.07312572 Inf   0.449  0.8949

Degrees-of-freedom method: asymptotic 
P value adjustment: tukey method for comparing a family of 3 estimates 
